# AnatomyLocked Character Studio

## Overview

This notebook is a **persistent human character studio** designed for:
- Reference-grade anatomy
- Identity-locked characters
- Regional anatomical refinement
- Pose-accurate deformation
- Reloadable, reusable characters
- Multi-character scene composition

This is **not** a random image generator.  
Once a character is finalized, they must only change in ways a real human could.

## Quick Identity Guide

**Must never change**
- Bone structure, facial geometry, body proportions
- Permanent skin features (freckles, moles, scars, birthmarks) and exact placement

**Allowed to vary**
- Pose/posture, muscle flexion/compression, skin folds due to movement
- Hair style (same root pattern unless explicitly changed)
- Makeup, nail color, lighting, camera angle, optional clothing

**Realistic deformation is expected**
- Stretching, bending, and flexing should change appearance only in ways a real body would

## Identity Lock Helper (Quick Settings)

- Fix the base seed for the character and keep it constant for identity-locked renders.
- Use the same identity embedding and reference set across all variants.
- Lock completed regions while tuning others (region-by-region refinement).
- Change only one factor at a time (pose, lighting, or camera) to isolate drift.

## Required Output Set (Reference Table)

| Set | Purpose | Views / Notes |
| --- | ------- | ------------- |
| Neutral | Baseline anatomy | Front, back, left, right |
| 3/4 | Shape consistency | Front 3/4 and back 3/4 |
| Poses | Deformation realism | Standing, sitting, crouched, dynamic |
| Lighting | Form clarity | Key, fill, rim; soft and hard |
| Scenes (opt) | Storytelling | Optional environment renders |

## Drift Triage Checklist

- Confirm base model, ControlNet stack, and seed match the baseline.
- Compare invariant features (moles, scars, facial proportions) against the baseline set.
- If drift appears, change only one variable at a time (pose, lighting, or camera).
- Re-run a neutral view to verify the lock before generating variants.
- If drift persists, refresh the identity embedding reference set.

In [ ]:
from pathlib import Path
import json

template_path = Path("Character_Validation_History_Template.json")

if template_path.exists():
    print(f"Template already exists: {template_path.resolve()}")
else:
    template = {
        "character_id": "CH-0001",
        "created_date": "YYYY-MM-DD",
        "baseline": {
            "base_model": "",
            "controlnet_stack": {
                "pose": "",
                "depth": "",
                "normal": ""
            },
            "seed": 0,
            "identity_embedding": {
                "type": "",
                "reference_images": []
            }
        },
        "identity_lock": {
            "locked_regions": ["face", "torso"],
            "notes": ""
        },
        "render_sets": [
            {
                "set_id": "SET-0001",
                "date": "YYYY-MM-DD",
                "purpose": "neutral views",
                "lighting": "",
                "camera": "",
                "pose_pack": "",
                "outputs": {
                    "image_paths": []
                },
                "validation": {
                    "identity_invariants_pass": False,
                    "anatomy_accuracy_pass": False,
                    "notes": ""
                }
            }
        ],
        "issues": [
            {
                "date": "YYYY-MM-DD",
                "issue": "",
                "resolution": ""
            }
        ],
        "next_steps": ""
    }
    template_path.write_text(json.dumps(template, indent=2), encoding="utf-8")
    print(f"Created template: {template_path.resolve()}")

In [ ]:
from datetime import date
from pathlib import Path
import json

template_path = Path("Character_Validation_History_Template.json")
output_dir = Path("character_histories")
output_dir.mkdir(exist_ok=True)

character_id = "CH-0001"
created_date = date.today().isoformat()

if not template_path.exists():
    print(f"Template not found: {template_path.resolve()}")
    print("Skipping history file creation. Add the template and re-run this cell.")
else:
    with template_path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)

    data["character_id"] = character_id
    data["created_date"] = created_date

    output_path = output_dir / f"{character_id}_history.json"
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)

    print(f"Wrote {output_path}")

---

## Core Rules

### Identity Invariants (Must Never Change)
- Bone structure
- Facial geometry
- Body proportions
- Permanent skin features (freckles, moles, scars, birthmarks)
- Relative placement and shape of invariant features

### Allowed Variations
- Pose and posture
- Muscle flexion and compression
- Skin folding due to movement
- Hair style (root pattern remains consistent)
- Makeup and nail color
- Lighting and camera angle
- Clothing (optional)

### Forbidden Variations
- Face drift
- Proportion changes
- Feature relocation
- Anatomy exaggeration
- Stylization that breaks realism




---

## Expected Output

Each character produces a reusable reference package:
- Neutral anatomy views
- Pose variations
- Lighting variants
- Scene renders
- Reloadable identity data

---

# SECTION 1 — Environment Setup & Dependencies

**Purpose:**  
Prepare Colab environment, GPU, and required libraries.

- Python version check
- GPU availability
- Dependency installation
- Cache and output directories

📌 1.1 Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


📌 1.2 Define AI Workspace Paths

In [ ]:
from pathlib import Path

# Base AI workspace
AI_BASE = Path("/content/drive/My Drive/AI")

# High-level directories
AI_DIRS = {
    "datasets": AI_BASE / "datasets",
    "experiments": AI_BASE / "Experiments",
    "images": AI_BASE / "Images",
    "rag": AI_BASE / "rag",
    "training": AI_BASE / "Training",
    "models": AI_BASE / "models",
}

# Model subdirectories (UNDER models/)
MODEL_DIRS = {
    "diffusion_base": AI_DIRS["models"] / "diffusion_base",
    "audio_models": AI_DIRS["models"] / "audio_models",
    "checkpoints": AI_DIRS["models"] / "checkpoints",
    "controlnet": AI_DIRS["models"] / "controlnet",
    "llm": AI_DIRS["models"] / "llm",
    "loras": AI_DIRS["models"] / "loras",
}


📌 1.3 Create Missing Directories (Non-Destructive)

In [ ]:
for name, path in AI_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"✓ {name}: {path}")

📌 1.4 Quick Sanity Check (Optional)

In [ ]:
assert AI_BASE.exists(), "AI base directory was not created correctly."
print("AI workspace is ready.")

🔍 1.5 Scan Existing Models & Assets

In [ ]:
def scan_directory(path, extensions=None):
    results = []
    if not path.exists():
        return results

    for p in path.rglob("*"):
        if p.is_file():
            if extensions is None or p.suffix.lower() in extensions:
                results.append(p)
    return results


📦 Scan Diffusion / ML Models

In [ ]:
MODEL_EXTS = {".ckpt", ".safetensors", ".pt", ".pth"}

base_models = scan_directory(MODEL_DIRS["diffusion_base"], MODEL_EXTS)
loras = scan_directory(MODEL_DIRS["loras"], MODEL_EXTS)
controlnets = scan_directory(MODEL_DIRS["controlnet"], MODEL_EXTS)
checkpoints = scan_directory(MODEL_DIRS["checkpoints"], MODEL_EXTS)


print(f"Base models: {len(base_models)}")
print(f"LoRAs: {len(loras)}")
print(f"ControlNets: {len(controlnets)}")
print(f"Checkpoints: {len(checkpoints)}")


📄 Optional: Print a Preview

In [ ]:
def preview(files, limit=10):
    for f in files[:limit]:
        print(f" - {f.name}")

print("\nSample base models:")
preview(base_models)

print("\nSample LoRAs:")
preview(loras)


---

# SECTION 2 — Base Model & Control Stack Selection

**Purpose:**  
Define the foundational models used throughout the notebook.

Includes:
- Base diffusion model (photorealistic, anatomy-capable)
- ControlNet modules (pose, depth, normals)
- Identity embedding models
- Version locking and rationale

This section should rarely change.

2.1 Model Registry Data Structures

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict


In [ ]:
@dataclass
class ModelEntry:
    name: str
    path: Path
    model_type: str     # e.g. "sd", "sdxl", "controlnet", "lora"
    size_mb: float


In [ ]:
MODEL_REGISTRY: Dict[str, List[ModelEntry]] = {
    "sd": [],
    "sdxl": [],
    "controlnet": [],
    "lora": [],
    "checkpoint": [],
    "llm": [],
    "audio": [],
}


🔍 2.2 Helper: File Size Utility

In [ ]:
def file_size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 ** 2), 2)


🔍 2.3 Heuristic Model Classifier

In [ ]:
def classify_model(path: Path) -> str:
    name = path.name.lower()

    if "controlnet" in name or "openpose" in name or "depth" in name or "normal" in name:
        return "controlnet"
    if "sdxl" in name or "sd_xl" in name or "sd xl" in name:
        return "sdxl"
    if "sd15" in name or "sd_1.5" in name or "sd-1.5" in name:
        return "sd"
    if "lora" in name or "lyco" in name:
        return "lora"
    if "audio" in name or "whisper" in name or "tts" in name:
        return "audio"
    if path.suffix in {".ckpt", ".safetensors"}:
        return "checkpoint"
    if path.suffix in {".bin", ".gguf"}:
        return "llm"

    return "unknown"


🔍 2.4 Scan Model Directories & Populate Registry

In [ ]:
MODEL_EXTS = {".ckpt", ".safetensors", ".pt", ".pth", ".bin", ".gguf"}


In [ ]:
def register_models_from_dir(directory: Path):
    for p in directory.rglob("*"):
        if p.is_file() and p.suffix.lower() in MODEL_EXTS:
            mtype = classify_model(p)
            if mtype in MODEL_REGISTRY:
                MODEL_REGISTRY[mtype].append(
                    ModelEntry(
                        name=p.name,
                        path=p,
                        model_type=mtype,
                        size_mb=file_size_mb(p)
                    )
                )


In [ ]:
# Scan model subdirectories only
for key in [
    "diffusion_base",
    "checkpoints",
    "controlnet",
    "loras",
    "llm",
    "audio_models",
]:
    register_models_from_dir(MODEL_DIRS[key])

📊 2.5 Registry Summary

In [ ]:
def print_registry_summary():
    print("📦 Model Registry Summary\n")
    for k, v in MODEL_REGISTRY.items():
        print(f"{k.upper():12s}: {len(v)} models")

print_registry_summary()


## Model Availability Snapshot (Run to Refresh)

Run the next cell to see what models and ControlNets are already available in this workspace.

In [ ]:
from IPython.display import Markdown, display

def build_registry_snapshot(limit: int = 5) -> str:
    rows = []
    for model_type, entries in MODEL_REGISTRY.items():
        if entries:
            names = ", ".join([e.name for e in entries[:limit]])
            if len(entries) > limit:
                names = f"{names} (+{len(entries) - limit} more)"
        else:
            names = "None"
        rows.append((model_type, len(entries), names))

    lines = ["| Type | Count | Sample |", "| --- | --- | --- |"]
    for model_type, count, names in rows:
        lines.append(f"| {model_type} | {count} | {names} |")
    return "\n".join(lines)

display(Markdown(build_registry_snapshot()))

In [ ]:
from pathlib import Path

snapshot_path = Path("model_registry_snapshot.md")
snapshot_path.write_text(build_registry_snapshot(), encoding="utf-8")
print(f"Wrote {snapshot_path}")

📄 2.6 Preview Models (Human-Readable)

In [ ]:
def preview_registry(model_type: str, limit: int = 10):
    entries = MODEL_REGISTRY.get(model_type, [])
    if not entries:
        print(f"No models registered for type: {model_type}")
        return

    print(f"\n{model_type.upper()} MODELS:")
    for e in entries[:limit]:
        print(f" - {e.name} ({e.size_mb} MB)")


In [ ]:
preview_registry("sd")
preview_registry("sdxl")
preview_registry("controlnet")
preview_registry("lora")

### Recommended Base + ControlNet Sources (SDXL + SD1.5)

Place files in the model folders shown below. Use the optional download section to fetch any URL.

- SDXL base -> models/diffusion_base
  https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
- SDXL ControlNet OpenPose -> models/controlnet
  https://huggingface.co/diffusers/controlnet-openpose-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors
- SDXL ControlNet Depth -> models/controlnet
  https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors
- SDXL ControlNet Normal -> models/controlnet
  https://huggingface.co/diffusers/controlnet-normal-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors

- SD1.5 base -> models/diffusion_base
  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
- SD1.5 ControlNet OpenPose -> models/controlnet
  https://huggingface.co/lllyasviel/control_v11p_sd15_openpose/resolve/main/diffusion_pytorch_model.safetensors
- SD1.5 ControlNet Depth -> models/controlnet
  https://huggingface.co/lllyasviel/control_v11f1p_sd15_depth/resolve/main/diffusion_pytorch_model.safetensors
- SD1.5 ControlNet Normal -> models/controlnet
  https://huggingface.co/lllyasviel/control_v11p_sd15_normalbae/resolve/main/diffusion_pytorch_model.safetensors


⭐ 2.7 Default Model Selection

In [ ]:
DEFAULT_MODELS = {
    "base_sd": None,
    "base_sdxl": None,
    "sd_controlnet_pose": None,
    "sd_controlnet_depth": None,
    "sd_controlnet_normal": None,
    "sdxl_controlnet_pose": None,
    "sdxl_controlnet_depth": None,
    "sdxl_controlnet_normal": None,
}


In [ ]:
def _detect_family_from_name(name: str) -> str:
    lowered = name.lower()
    if "sdxl" in lowered or "sd_xl" in lowered or "sd xl" in lowered:
        return "sdxl"
    if "sd15" in lowered or "sd_1.5" in lowered or "sd-1.5" in lowered:
        return "sd"
    return "unknown"

def select_default(model_type: str, contains: str, family: str | None = None):
    for m in MODEL_REGISTRY.get(model_type, []):
        if contains.lower() in m.name.lower():
            if family:
                m_family = _detect_family_from_name(m.name)
                if m_family != family:
                    continue
            return m
    return None


Example defaults (adjust names as needed)

In [ ]:
DEFAULT_MODELS["base_sdxl"] = select_default("sdxl", "base")
DEFAULT_MODELS["base_sd"] = select_default("sd", "v1-5")

DEFAULT_MODELS["sdxl_controlnet_pose"] = select_default("controlnet", "openpose", "sdxl")
DEFAULT_MODELS["sdxl_controlnet_depth"] = select_default("controlnet", "depth", "sdxl")
DEFAULT_MODELS["sdxl_controlnet_normal"] = select_default("controlnet", "normal", "sdxl")

DEFAULT_MODELS["sd_controlnet_pose"] = select_default("controlnet", "openpose", "sd")
DEFAULT_MODELS["sd_controlnet_depth"] = select_default("controlnet", "depth", "sd")
DEFAULT_MODELS["sd_controlnet_normal"] = select_default("controlnet", "normal", "sd")

ACTIVE_BASE_FAMILY = "sdxl" if DEFAULT_MODELS["base_sdxl"] else "sd" if DEFAULT_MODELS["base_sd"] else None

if ACTIVE_BASE_FAMILY == "sdxl":
    ACTIVE_CONTROLNET_DEFAULTS = {
        "pose": DEFAULT_MODELS["sdxl_controlnet_pose"],
        "depth": DEFAULT_MODELS["sdxl_controlnet_depth"],
        "normal": DEFAULT_MODELS["sdxl_controlnet_normal"],
    }
elif ACTIVE_BASE_FAMILY == "sd":
    ACTIVE_CONTROLNET_DEFAULTS = {
        "pose": DEFAULT_MODELS["sd_controlnet_pose"],
        "depth": DEFAULT_MODELS["sd_controlnet_depth"],
        "normal": DEFAULT_MODELS["sd_controlnet_normal"],
    }
else:
    ACTIVE_CONTROLNET_DEFAULTS = {"pose": None, "depth": None, "normal": None}

In [ ]:
print("⭐ Default Model Selection\n")
for k, v in DEFAULT_MODELS.items():
    print(f"{k:22s}: {v.name if v else 'None'}")

print("\nActive base family:", ACTIVE_BASE_FAMILY or "None")

In [ ]:
def _detect_family(name: str) -> str:
    lowered = name.lower()
    if "sdxl" in lowered or "sd_xl" in lowered or "sd xl" in lowered:
        return "sdxl"
    if "sd15" in lowered or "sd_1.5" in lowered or "sd-1.5" in lowered:
        return "sd"
    return "unknown"

def warn_controlnet_compatibility(base_entry, control_entries):
    if base_entry is None:
        print("Base model not set. Skipping compatibility checks.")
        return

    base_family = _detect_family(base_entry.name)
    if base_family == "unknown":
        print(f"Base model family unknown: {base_entry.name}")
        return

    for key, entry in control_entries.items():
        if entry is None:
            continue
        control_family = _detect_family(entry.name)
        if control_family == "unknown":
            print(f"ControlNet {key} family unknown: {entry.name}")
            continue
        if control_family != base_family:
            print(
                f"⚠️ ControlNet {key} looks like {control_family} but base is {base_family}. "
                "Expect poor results or load errors."
            )

print("\nCompatibility checks:")
warn_controlnet_compatibility(
    DEFAULT_MODELS.get("base_sdxl"),
    {
        "pose": DEFAULT_MODELS.get("sdxl_controlnet_pose"),
        "depth": DEFAULT_MODELS.get("sdxl_controlnet_depth"),
        "normal": DEFAULT_MODELS.get("sdxl_controlnet_normal"),
    }
)
warn_controlnet_compatibility(
    DEFAULT_MODELS.get("base_sd"),
    {
        "pose": DEFAULT_MODELS.get("sd_controlnet_pose"),
        "depth": DEFAULT_MODELS.get("sd_controlnet_depth"),
        "normal": DEFAULT_MODELS.get("sd_controlnet_normal"),
    }
)

# **🔹 SECTION 2.8 — Diffusers Pipeline Initialization (Registry-Driven)**

**2.8.1 Install Diffusers Stack (Once)**

To ensure a `pip install` command runs only if necessary, you can check for the presence of a key module before executing the installation. This makes your notebook more efficient and prevents unnecessary re-installations. Here's an example using the `diffusers` library:

In [ ]:
try:
    import diffusers
    print("diffusers is already installed.")
except ImportError:
    print("diffusers not found, installing...")
    !pip install -q diffusers transformers accelerate xformers safetensors
    print("Installation complete.")

**2.8.2 Imports & Accelerator Setup**

In [ ]:
import torch
from diffusers import (
    StableDiffusionXLPipeline,
    StableDiffusionXLControlNetPipeline,
    ControlNetModel
)
from diffusers.utils import load_image


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Using device: {DEVICE}, dtype: {DTYPE}")


**2.8.3 Resolve Base Model from Registry**

In [ ]:
def require_model(entry, label):
    if entry is None:
        raise RuntimeError(f"Required model missing: {label}")
    return str(entry.path)

In [ ]:
BASE_SDXL_PATH = require_model(
    DEFAULT_MODELS["base_sdxl"],
    "Base SDXL model"
)

**2.8.4 Load ControlNet Models (If Available)**

In [ ]:
CONTROLNETS = {}

def load_controlnet(key, model_entry):
    if model_entry is None:
        print(f"⚠️ ControlNet {key} not found — skipping")
        return None

    print(f"Loading ControlNet: {model_entry.name}")
    return ControlNetModel.from_single_file(
        model_entry.path,
        torch_dtype=DTYPE
    )

In [ ]:
CONTROLNETS["pose"] = load_controlnet(
    "pose",
    ACTIVE_CONTROLNET_DEFAULTS.get("pose")
 )

CONTROLNETS["depth"] = load_controlnet(
    "depth",
    ACTIVE_CONTROLNET_DEFAULTS.get("depth")
 )

CONTROLNETS["normal"] = load_controlnet(
    "normal",
    ACTIVE_CONTROLNET_DEFAULTS.get("normal")
 )

In [ ]:
ACTIVE_CONTROLNETS = [cn for cn in CONTROLNETS.values() if cn is not None]
print(f"Active ControlNets: {len(ACTIVE_CONTROLNETS)}")


**2.8.5 Initialize the Pipeline**

In [ ]:
if ACTIVE_CONTROLNETS:
    pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
        BASE_SDXL_PATH,
        controlnet=ACTIVE_CONTROLNETS,
        torch_dtype=DTYPE,
        safety_checker=None,
        variant="fp16"
    )
else:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        BASE_SDXL_PATH,
        torch_dtype=DTYPE,
        safety_checker=None,
        variant="fp16"
    )


In [ ]:
pipe.to(DEVICE)
pipe.enable_xformers_memory_efficient_attention()
pipe.enable_model_cpu_offload()


**2.8.6 Deterministic Seeding (Critical for Identity Work)**

In [ ]:
import random
import numpy as np

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


**2.8.7 Smoke Test (Minimal Render)**

In [ ]:
set_seed(12345)

prompt = (
    "neutral anatomical reference photograph of an adult human, "
    "standing relaxed, evenly lit, realistic proportions"
)

image = pipe(
    prompt=prompt,
    num_inference_steps=30,
    guidance_scale=5.5
).images[0]

display(image)


---

# SECTION 3 — Exploration Mode (Free Variation Playground)

**Purpose:**  
Generate broad anatomical variations to discover promising candidates.

Characteristics:
- Controlled randomness
- No identity locking
- Multiple seeds
- Neutral reference lighting

**Output:**  
A set of candidate images for review.

### Prompt Additions (Optional)

Add extra descriptors separated by commas (e.g., "athletic build, short hair").

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets not found, installing...")
    !pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display

prompt_additions_text = widgets.Textarea(
    value="",
    placeholder="Comma-separated additions for the prompt...",
    description="Prompt +:",
    layout=widgets.Layout(width="80%", height="80px")
)

negative_additions_text = widgets.Textarea(
    value="",
    placeholder="Comma-separated additions for the negative prompt...",
    description="Negative +:",
    layout=widgets.Layout(width="80%", height="80px")
)

run_button = widgets.Button(description="Run", button_style="primary")
run_output = widgets.Output()

PROMPT_ADDITIONS = []
NEGATIVE_PROMPT_ADDITIONS = []

def _parse_additions(text: str):
    return [item.strip() for item in text.split(",") if item.strip()]

def _run_clicked(_):
    global PROMPT_ADDITIONS, NEGATIVE_PROMPT_ADDITIONS
    PROMPT_ADDITIONS = _parse_additions(prompt_additions_text.value)
    NEGATIVE_PROMPT_ADDITIONS = _parse_additions(negative_additions_text.value)
    with run_output:
        run_output.clear_output()
        print("Prompt additions set for next run.")
        print("Negative additions set for next run.")
    prompt_additions_text.value = ""
    negative_additions_text.value = ""

run_button.on_click(_run_clicked)
display(widgets.VBox([prompt_additions_text, negative_additions_text, run_button, run_output]))

In [ ]:
from datetime import datetime
from pathlib import Path
from IPython.display import display

EXPLORATION_DIR = AI_DIRS["images"] / "exploration" / datetime.utcnow().strftime("%Y%m%d_%H%M%S")
EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)

base_prompt = (
    "neutral anatomical reference photograph of an adult human, "
    "standing relaxed, evenly lit, realistic proportions"
 )
base_negative_prompt = (
    "stylized, cartoon, exaggerated anatomy, deformed, extra limbs, "
    "low quality, blurry, lowres"
 )

prompt_additions = ", ".join(PROMPT_ADDITIONS) if "PROMPT_ADDITIONS" in globals() else ""
negative_additions = ", ".join(NEGATIVE_PROMPT_ADDITIONS) if "NEGATIVE_PROMPT_ADDITIONS" in globals() else ""

prompt = base_prompt + (", " + prompt_additions if prompt_additions else "")
negative_prompt = base_negative_prompt + (", " + negative_additions if negative_additions else "")

seeds = [1001, 1002, 1003, 1004]
num_steps = 30
guidance_scale = 5.5

results = []
for seed in seeds:
    set_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale
    ).images[0]
    filename = f"candidate_seed_{seed}.png"
    out_path = EXPLORATION_DIR / filename
    image.save(out_path)
    results.append((seed, out_path, image))
    print(f"Saved {out_path}")

for seed, path, image in results:
    display(image)

PROMPT_ADDITIONS = []
NEGATIVE_PROMPT_ADDITIONS = []

In [ ]:
from PIL import Image

if not results:
    print("No exploration results found.")
else:
    cols = 2
    rows = (len(results) + cols - 1) // cols
    w, h = results[0][2].size
    grid = Image.new("RGB", (w * cols, h * rows), (0, 0, 0))
    for idx, (_, _, img) in enumerate(results):
        r = idx // cols
        c = idx % cols
        grid.paste(img, (c * w, r * h))

    grid_path = EXPLORATION_DIR / "grid.png"
    grid.save(grid_path)
    display(grid)
    print(f"Saved {grid_path}")

In [ ]:
import csv

csv_path = EXPLORATION_DIR / "candidates.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "seed",
            "image_path",
            "prompt",
            "negative_prompt",
            "num_steps",
            "guidance_scale"
        ]
    )
    writer.writeheader()
    for seed, path, _ in results:
        writer.writerow({
            "seed": seed,
            "image_path": str(path),
            "prompt": prompt,
            "negative_prompt": negative_prompt,
            "num_steps": num_steps,
            "guidance_scale": guidance_scale
        })

print(f"Wrote {csv_path}")

---

# SECTION 4 — Candidate Selection & Identity Capture

**Purpose:**  
Promote one exploration image into a persistent character.

Steps:
- Select a single candidate image
- Assign a character name / ID
- Extract face embedding
- Capture base seed
- Initialize character directory

**Result:**  
A named character exists for the first time.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil

CHARACTER_ROOT = AI_DIRS["datasets"] / "characters"
CHARACTER_ROOT.mkdir(parents=True, exist_ok=True)

def init_character_from_candidate(
    candidate_image_path: str,
    character_id: str,
    base_seed: int,
    notes: str = "",
    face_embedding = None
    ) -> Path:
    candidate_path = Path(candidate_image_path).expanduser()
    if not candidate_path.exists():
        raise FileNotFoundError(f"Candidate image not found: {candidate_path}")

    character_dir = CHARACTER_ROOT / character_id
    if character_dir.exists():
        raise FileExistsError(f"Character already exists: {character_dir}")
    character_dir.mkdir(parents=True, exist_ok=False)

    candidate_name = f"candidate{candidate_path.suffix}"
    candidate_dest = character_dir / candidate_name
    shutil.copy2(candidate_path, candidate_dest)

    embedding_status = "pending"
    embedding_dim = None
    embedding_data = None
    if face_embedding is not None:
        embedding_status = "captured"
        embedding_data = list(face_embedding)
        embedding_dim = len(embedding_data)

    metadata = {
        "character_id": character_id,
        "created_at": datetime.utcnow().isoformat() + "Z",
        "candidate_image": candidate_name,
        "base_seed": int(base_seed),
        "notes": notes,
        "embedding_status": embedding_status,
        "embedding_dim": embedding_dim,
        "embedding": embedding_data
    }

    (character_dir / "character.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8"
    )
    (character_dir / "seed.txt").write_text(str(base_seed), encoding="utf-8")

    print(f"Initialized character at {character_dir}")
    return character_dir

In [ ]:
candidate_image_path = ""  # e.g., /content/drive/My Drive/AI/Images/candidate_001.png
character_id = "CH-0001"
base_seed = 0
notes = ""

if not candidate_image_path:
    print("Set candidate_image_path and re-run this cell.")
else:
    init_character_from_candidate(
        candidate_image_path=candidate_image_path,
        character_id=character_id,
        base_seed=base_seed,
        notes=notes
    )


---

# SECTION 5 — Identity Locking & Invariant Feature Definition

**Purpose:**  
Define what can never change for this character.

Includes:
- Facial geometry lock
- Body proportion constraints
- Invariant skin feature masks
- Identity validation checks

From this point forward, the character is identity-locked.

---

# SECTION 6 — Body Region Map Creation

**Purpose:**  
Divide the body into anatomically meaningful regions for refinement.

Example regions:
- Head
- Neck
- Shoulders
- Chest
- Abdomen
- Hips
- Thighs
- Calves
- Upper arms
- Forearms
- Hands
- Feet

Each region receives a mask or index.

---

# SECTION 7 — Regional Refinement Mode (Anatomical Sculpting)

**Purpose:**  
Iteratively refine anatomy one region at a time.

Workflow:
1. Lock entire body
2. Unlock one region
3. Adjust within realistic bounds
4. Freeze region permanently
5. Move to next region

This mirrors real sculpting discipline.

---

# SECTION 8 — Canonical Body Finalization

**Purpose:**  
Declare the anatomy complete.

Includes:
- Full-body lock
- Cross-pose consistency checks
- Identity snapshot
- Version tagging

After this point, anatomy should not be altered.

---

# SECTION 9 — Pose & Deformation Generation

**Purpose:**  
Generate realistic pose-driven deformation.

Examples:
- Standing
- Sitting
- Walking
- Reaching
- Twisting
- Bending
- Flexing

Rules:
- Deformation allowed
- Identity drift forbidden

---

# SECTION 10 — Lighting, Camera & Reference Views

**Purpose:**  
Create artist-friendly reference images.

Includes:
- Neutral studio lighting
- Orthographic-like views
- Camera angle sweeps
- Optional dramatic lighting

---

# SECTION 11 — Reference Set Export

**Purpose:**  
Export a complete, organized reference set.

Includes:
- Consistent filenames
- Pose labeling
- Lighting variants
- Metadata files

This output is ready for drawing, modeling, or study.

---

# SECTION 12 — Character Save & Reload System

**Purpose:**  
Persist characters across sessions.

Capabilities:
- Save character state
- Reload identity locks
- Resume rendering
- Continue refinement (if unlocked)

Characters become reusable assets.

---

# SECTION 13 — Scene Rendering (Single Character)

**Purpose:**  
Place a character into an environment while preserving identity.

Examples:
- Sitting on a bench
- Walking through a park
- Standing in conversation
- Environmental interaction

---

# SECTION 14 — Multi-Character Scene Composition

**Purpose:**  
Create scenes involving multiple characters.

Workflow:
1. Load characters independently
2. Generate poses separately
3. Match camera and lighting
4. Composite using depth awareness

Example:
- Character B sitting
- Character F approaching and waving

---

# SECTION 15 — Quality Control & Drift Detection

**Purpose:**  
Ensure long-term identity stability.

Includes:
- Embedding similarity checks
- Visual diffs
- Automated rejection rules
- Manual inspection tools

---

# SECTION 16 — Archive, Export & Versioning

**Purpose:**  
Long-term character management.

Includes:
- Version history
- Anatomy revisions
- Export formats
- Backup strategy

---

# SECTION 17 — Notes, Experiments & Future Extensions

**Purpose:**  
A sandbox for:
- Model upgrades
- New controls
- Experimental ideas
- Deferred features

This section keeps the rest of the notebook clean.

<details>
<summary><strong>SECTION 3 — Model Downloading and Management (Optional)</strong></summary>

Use this section only when you need to add new models.

**Optional section**: Model downloads are disabled by default. Set `ENABLE_MODEL_DOWNLOADS = True` in the next cell to enable.

In [ ]:
ENABLE_MODEL_DOWNLOADS = False
print(f"Model downloads enabled: {ENABLE_MODEL_DOWNLOADS}")

### Tokens (Optional)

- `HF_TOKEN` for private or gated Hugging Face models.
- `CIVIT_TOKEN` for authenticated Civitai downloads.

In [ ]:
if not ENABLE_MODEL_DOWNLOADS:
    print("Model downloads disabled. Skipping install.")
else:
    try:
        import huggingface_hub
        print("huggingface_hub is already installed.")
    except ImportError:
        print("huggingface_hub not found, installing huggingface_hub, gdown, and requests...")
        !pip install -q huggingface_hub gdown requests
        print("Installation complete.")

In [ ]:
from google.colab import userdata

def get_hf_token() -> str:
    """
    Retrieves the Hugging Face token from Colab's secrets manager.
    Returns an empty string if the token is not found.
    """
    try:
        token = userdata.get('HF_TOKEN')
        if token:
            print("✓ Hugging Face token successfully retrieved from Colab secrets.")
            return token
        else:
            print("✗ Hugging Face token (HF_TOKEN) not found in Colab secrets. Authenticated downloads may fail.")
            return ""
    except userdata.SecretNotFoundError:
        print("✗ Hugging Face token (HF_TOKEN) not found in Colab secrets. Authenticated downloads may fail.")
        return ""
    except Exception as e:
        print(f"✗ An error occurred while retrieving Hugging Face token: {e}")
        return ""

print("get_hf_token function defined.")

In [ ]:
from google.colab import userdata

def get_civit_token() -> str:
    """
    Retrieves the Civitai token from Colab's secrets manager.
    Returns an empty string if the token is not found.
    """
    try:
        token = userdata.get('CIVIT_TOKEN')
        if token:
            print("✓ Civitai token successfully retrieved from Colab secrets.")
            return token
        else:
            print("✗ Civitai token (CIVIT_TOKEN) not found in Colab secrets. Authenticated Civitai downloads may fail.")
            return ""
    except userdata.SecretNotFoundError:
        print("✗ Civitai token (CIVIT_TOKEN) not found in Colab secrets. Authenticated Civitai downloads may fail.")
        return ""
    except Exception as e:
        print(f"✗ An error occurred while retrieving Civitai token: {e}")
        return ""

print("get_civit_token function defined.")

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import requests
import gdown
from tqdm.auto import tqdm

_HF_HUB_AVAILABLE = False
if ENABLE_MODEL_DOWNLOADS:
    try:
        import huggingface_hub
        from huggingface_hub import hf_hub_download
        from huggingface_hub.utils import HfHubHTTPError
        _HF_HUB_AVAILABLE = True
        print("✓ huggingface_hub download utilities are available.")
    except ImportError as e:
        _HF_HUB_AVAILABLE = False
        print(f"✗ huggingface_hub not available: {e}. HF links will fall back to direct download.")
    except Exception as e:
        _HF_HUB_AVAILABLE = False
        print(f"✗ Hugging Face setup error: {e}. HF links will fall back to direct download.")
else:
    print("Model downloads disabled. Skipping huggingface_hub setup.")

def _is_hf_url(parsed_url) -> bool:
    host = parsed_url.netloc.lower()
    return "huggingface.co" in host or "hf.co" in host

def download_file(url: str, destination_path: Path) -> bool:
    """
    Downloads a file from a given URL to a specified local path.
    Handles direct HTTP/HTTPS downloads, Google Drive links, and Hugging Face URLs.
    """
    print(f"Attempting to download from {url} to {destination_path}")
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        parsed_url = urlparse(url)

        # 1. Hugging Face URLs
        if _HF_HUB_AVAILABLE and _is_hf_url(parsed_url):
            print("Detected Hugging Face link.")
            hf_token = get_hf_token()

            path_parts = parsed_url.path.split('/')
            if len(path_parts) >= 5 and path_parts[3] == 'resolve':
                repo_id = f"{path_parts[1]}/{path_parts[2]}"
                filename = path_parts[-1]
            else:
                print("Could not parse Hugging Face repo_id and filename from URL. Attempting direct download.")
                return download_file_direct(url, destination_path)

            if not filename:
                print("Could not extract filename from Hugging Face URL. Attempting direct download.")
                return download_file_direct(url, destination_path)

            print(f"HF: repo_id={repo_id}, filename={filename}")
            try:
                downloaded_path = hf_hub_download(
                    repo_id=repo_id,
                    filename=filename,
                    local_dir=destination_path.parent,
                    local_dir_use_symlinks=False,
                    token=hf_token if hf_token else None
                )
                if Path(downloaded_path).name != destination_path.name:
                    Path(downloaded_path).rename(destination_path)
                print(f"✓ Successfully downloaded {destination_path.name} from Hugging Face.")
                return True
            except HfHubHTTPError as e:
                print(f"✗ Error during Hugging Face download (HTTP Error): {e}")
                if '401 Client Error' in str(e) or '403 Client Error' in str(e):
                    print("    This might be a private model or require authentication. Check your HF_TOKEN.")
                if destination_path.exists():
                    destination_path.unlink()
                return False
            except Exception as e:
                print(f"✗ Hugging Face Hub operation failed or is disabled: {e}. Attempting direct download as fallback.")
                return download_file_direct(url, destination_path)

        # 2. Google Drive links
        if "drive.google.com" in url:
            print("Detected Google Drive link.")
            gdown.download(url, str(destination_path), quiet=False, fuzzy=True)
            print(f"✓ Successfully downloaded {destination_path.name}")
            return True

        # 3. Direct HTTP/HTTPS downloads (including Civitai)
        return download_file_direct(url, destination_path)
    except gdown.exceptions.GDriveDownloadError as e:
        print(f"✗ Error during Google Drive download: {e}")
        if destination_path.exists():
            destination_path.unlink()
        return False
    except Exception as e:
        print(f"✗ An unexpected error occurred: {e}")
        if destination_path.exists():
            destination_path.unlink()
        return False

def download_file_direct(url: str, destination_path: Path) -> bool:
    """
    Helper function for direct HTTP/HTTPS downloads.
    """
    print("Attempting direct download.")
    headers = {}
    if "civitai.com" in url:
        civit_token = get_civit_token()
        if civit_token:
            headers["Authorization"] = f"Bearer {civit_token}"
            print("Added Civitai token to headers for authenticated download.")
        else:
            print("Civitai token not found, attempting unauthenticated download.")

    try:
        response = requests.get(url, stream=True, headers=headers)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))

        with open(destination_path, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=destination_path.name) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        return True
    except requests.exceptions.RequestException as e:
        print(f"✗ Error during direct download: {e}")
        if destination_path.exists():
            destination_path.unlink()
        return False

print("Download utilities ready.")

In [ ]:
from typing import Optional

def smart_download_model(url: str, model_type_hint: Optional[str] = None) -> bool:
    """
    Downloads a model from a URL to the correct subdirectory within AI_DIRS['models']
    and registers it in the MODEL_REGISTRY.
    """
    if not ENABLE_MODEL_DOWNLOADS:
        print("Model downloads disabled. Set ENABLE_MODEL_DOWNLOADS = True to enable.")
        return False

    filename = Path(url).name
    print(f"Attempting to download model: {filename}")

    model_type = model_type_hint.lower() if model_type_hint else classify_model(Path(filename))
    target_dir = AI_DIRS["models"]
    if model_type == "controlnet":
        target_dir = MODEL_DIRS["controlnet"]
    elif model_type == "lora":
        target_dir = MODEL_DIRS["loras"]
    elif model_type == "llm":
        target_dir = MODEL_DIRS["llm"]
    elif model_type == "audio":
        target_dir = MODEL_DIRS["audio_models"]
    elif model_type == "checkpoint":
        target_dir = MODEL_DIRS["checkpoints"]

    destination_path = target_dir / filename
    print(f"Deduced model type: {model_type}, Target path: {destination_path}")

    if destination_path.exists():
        print(f"Model already exists at {destination_path}. Skipping download.")
        download_successful = True
    else:
        download_successful = download_file(url, destination_path)

    if download_successful:
        print("Download successful (or file already existed). Updating model registry...")
        for k in MODEL_REGISTRY:
            MODEL_REGISTRY[k].clear()
        register_models_from_dir(AI_DIRS["models"])
        print("Model registry updated.")
        return True

    print(f"Failed to download model from {url}.")
    return False

### Quick Download (Paste URL)

Paste a model URL and (optionally) a type hint, then run the next cell.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets not found, installing...")
    !pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, clear_output

url_text = widgets.Text(
    value="",
    placeholder="Paste a model URL...",
    description="URL:",
    layout=widgets.Layout(width="80%")
)

type_dropdown = widgets.Dropdown(
    options=["auto", "controlnet", "lora", "sdxl", "sd", "checkpoint", "llm", "audio"],
    value="auto",
    description="Type:",
    layout=widgets.Layout(width="50%")
)

download_button = widgets.Button(description="Download", button_style="primary")
output = widgets.Output()

def on_download_clicked(_):
    with output:
        clear_output()
        if not ENABLE_MODEL_DOWNLOADS:
            print("Model downloads are disabled. Set ENABLE_MODEL_DOWNLOADS = True and re-run this cell.")
            return
        url = url_text.value.strip()
        if not url:
            print("Paste a model URL into the URL field.")
            return
        hint = None if type_dropdown.value == "auto" else type_dropdown.value
        smart_download_model(url, model_type_hint=hint)

download_button.on_click(on_download_clicked)
display(widgets.VBox([url_text, type_dropdown, download_button, output]))

</details>